# Chapter 2 — NB 02: comparação do MAF do ZNF175 entre v1 (Park) e v2 (Hui)

**Pergunta:** o **substrato de variantes** do ZNF175 é estável entre v1 (Freeze One, 11.451) e v2 (Release 2020 v2.0, 43.731)?
Variantes que **somem** ou cujo **MAF muda muito** são candidatos a **artefato técnico** — o que a Molly pediu para descartar antes de invocar biologia.

**Pipeline simétrico (independente de Daniel/Park):** ambas as coortes extraídas pelo **mesmo** caminho —
`bcftools view -r 19:51571283-51592510` (janela NCBI) → `norm -f <GRCh38.fa> -a -m-any` (**left-align + atomize** + split) → `--set-id chr:pos:ref:alt` → `plink2 --freq --geno-counts`.
A atomização (`-a`) decompõe MNVs em SNVs atômicas (ex.: `ATT:ACT` → `T:C`), garantindo que a **mesma variante recebe a mesma chave** nas duas coortes.
Jobs LSF: `scripts/extract_znf175_region.sh` + `submit_v1_znf175.sh` / `submit_v2_znf175.sh`.

| | v1 (Park / Freeze One) | v2 (Hui / Release 2020 v2.0) |
|---|---|---|
| prefixo | `results/v1_znf175_strict` | `results/v2_znf175_release` |
| N | 11.451 | 43.731 |

> ⚠️ **Caveat:** o MAF aqui é **da coorte inteira** (não estratificado por ancestralidade). Diferenças podem vir de (1) pipeline/técnico, (2) crescimento da coorte, (3) composição de ancestralidade. A estratificação por ancestralidade é o próximo refinamento. Aqui focamos em **presença/ausência** e **deslocamentos grandes de MAF**.

In [1]:
# --- Setup + loader ---
from pathlib import Path
import pandas as pd
import numpy as np

BASE    = Path("/project/hall/analysis/hearing-loss-genomics")
RESULTS = BASE / "analysis/chapter_2/results/02"   # this block's outputs
RESULTS01 = BASE / "analysis/chapter_2/results/01" # NB01 outputs (for cross-check)

def load_freq(prefix):
    # Read plink2 .afreq + .gcount into one table with MAF, carriers, AN and a (pos:ref:alt) key.
    af = pd.read_csv(f"{prefix}.afreq", sep="\t")[["#CHROM","ID","REF","ALT","ALT_FREQS","OBS_CT"]]
    gc = pd.read_csv(f"{prefix}.gcount", sep="\t")[
            ["#CHROM","ID","REF","ALT","HOM_REF_CT","HET_REF_ALT_CTS","TWO_ALT_GENO_CTS","MISSING_CT"]]
    df = af.merge(gc, on=["#CHROM","ID","REF","ALT"])
    df["POS"]       = df["ID"].str.split(":").str[1].astype(int)
    df["ALT_FREQS"] = pd.to_numeric(df["ALT_FREQS"], errors="coerce")
    df["MAF"]       = df["ALT_FREQS"].apply(lambda x: min(x, 1-x) if pd.notna(x) else np.nan)
    df["carriers"]  = df["HET_REF_ALT_CTS"] + df["TWO_ALT_GENO_CTS"]
    df["AC_alt"]    = df["HET_REF_ALT_CTS"] + 2*df["TWO_ALT_GENO_CTS"]
    df["AN"]        = 2*(df["HOM_REF_CT"] + df["HET_REF_ALT_CTS"] + df["TWO_ALT_GENO_CTS"])
    df["key"]       = df["POS"].astype(str) + ":" + df["REF"] + ":" + df["ALT"]
    df["is_indel"]  = (df["REF"].str.len() != 1) | (df["ALT"].str.len() != 1)
    return df.drop_duplicates("key").reset_index(drop=True)

v1 = load_freq(RESULTS / "v1_znf175_strict")
v2 = load_freq(RESULTS / "v2_znf175_release")
print(f"v1 variants: {len(v1)}  (pos {v1.POS.min():,}-{v1.POS.max():,})")
print(f"v2 variants: {len(v2)}  (pos {v2.POS.min():,}-{v2.POS.max():,})")

v1 variants: 161  (pos 51,573,255-51,588,539)
v2 variants: 380  (pos 51,573,230-51,588,560)


## 0. Overview por-coorte (simétrico ao NB 01)

Antes de comparar, o resumo standalone de **cada** coorte: total de variantes, quantas raras (MAF<0,1%) e quantas comuns (MAF≥1%).

In [2]:
# --- Per-cohort standalone overview (symmetric to NB 01's v1 summary) ---
def overview(df, label):
    n = len(df)
    rare   = int((df["MAF"] < 0.001).sum())
    common = int((df["MAF"] >= 0.01).sum())
    print(f"{label:30s} total={n:4d}  MAF<0.1%={rare:4d}  MAF>=1%={common}")
    return {"cohort": label, "total": n, "rare_lt_0.1pct": rare, "common_ge_1pct": common}

ov = pd.DataFrame([
    overview(v1, "v1 (Park / Freeze One, 11451)"),
    overview(v2, "v2 (Hui / Release 2.0, 43731)"),
])
ov.to_csv(RESULTS / "per_cohort_overview.csv", index=False)
print("\nNote: the 8 common (MAF>=1%) variants are expected to match across freezes (stable backbone);")
print("rare counts scale with cohort size (more samples -> more rare variants observed).")

v1 (Park / Freeze One, 11451)  total= 161  MAF<0.1%= 142  MAF>=1%=8
v2 (Hui / Release 2.0, 43731)  total= 380  MAF<0.1%= 356  MAF>=1%=8

Note: the 8 common (MAF>=1%) variants are expected to match across freezes (stable backbone);
rare counts scale with cohort size (more samples -> more rare variants observed).


## 1. Matching por `(pos, ref, alt)` e presença em cada coorte

Juntamos os dois conjuntos pela chave normalizada. Três grupos:
- **shared** — presente nas duas (dá para comparar MAF)
- **v1-only** — presente no v1 (11K) e **ausente no v2** (44K) → ⚠️ **suspeito** (sumiu apesar de 4× mais amostras)
- **v2-only** — presente só no v2 → em geral **esperado** (mais amostras revelam mais variantes raras)

In [3]:
# --- Merge & classify ---
m = v1.merge(v2, on="key", how="outer", suffixes=("_v1","_v2"), indicator=True)
grp = m["_merge"].map({"both":"shared","left_only":"v1-only","right_only":"v2-only"})
print(grp.value_counts().to_string())

shared  = m[m["_merge"]=="both"].copy()
v1_only = m[m["_merge"]=="left_only"].copy()
v2_only = m[m["_merge"]=="right_only"].copy()

# indel breakdown of the non-shared groups (left-align caveat is about indels)
def indel_split(d, col):
    s = d[col].fillna(False)
    return f"SNV={int((~s).sum())}, indel={int(s.sum())}"
print("\nv1-only:", indel_split(v1_only, "is_indel_v1"), " <-- check these (disappeared in v2)")
print("v2-only:", indel_split(v2_only, "is_indel_v2"), " <-- mostly expected (bigger cohort)")

_merge
v2-only    225
shared     155
v1-only      6

v1-only: SNV=-6, indel=0  <-- check these (disappeared in v2)
v2-only: SNV=-239, indel=14  <-- mostly expected (bigger cohort)


## 1b. União de todas as variantes — a "visão do todo"

Tabela com **todas** as variantes (união v1 ∪ v2). Quando uma variante existe só num freeze, as colunas do outro ficam **em branco**.
Formato no espírito do `cmp_shared_maf.csv`. Salvo em `cmp_union_all_variants.csv`.

In [4]:
# --- Union table: ALL variants across v1 and v2; blanks where absent in a cohort ---
u = m.copy()
parts = u["key"].str.split(":", expand=True)   # key = POS:REF:ALT
u["CHROM"] = "19"
u["POS"]   = parts[0].astype(int)
u["REF"]   = parts[1]
u["ALT"]   = parts[2]
u["group"]      = u["_merge"].map({"both":"shared","left_only":"v1_only","right_only":"v2_only"})
u["present_v1"] = np.where(u["_merge"] != "right_only", "Y", "N")
u["present_v2"] = np.where(u["_merge"] != "left_only",  "Y", "N")
u["is_indel"]   = u["is_indel_v1"].fillna(u["is_indel_v2"])
u["MAF_ratio"]  = u["MAF_v2"] / u["MAF_v1"]

union_cols = ["key","CHROM","POS","REF","ALT","group","present_v1","present_v2",
              "MAF_v1","MAF_v2","carriers_v1","carriers_v2","AC_alt_v1","AC_alt_v2",
              "AN_v1","AN_v2","is_indel","MAF_ratio"]
u = u[union_cols].sort_values("POS").reset_index(drop=True)
u.to_csv(RESULTS / "cmp_union_all_variants.csv", index=False)

print(f"union total: {len(u)}  "
      f"(shared={(u.group=='shared').sum()}, "
      f"v1_only={(u.group=='v1_only').sum()}, "
      f"v2_only={(u.group=='v2_only').sum()})")
print("\npreview (blanks = absent in that cohort):")
print(u.head(12).to_string(index=False))

union total: 386  (shared=155, v1_only=6, v2_only=225)

preview (blanks = absent in that cohort):
                 key CHROM      POS       REF ALT   group present_v1 present_v2   MAF_v1   MAF_v2  carriers_v1  carriers_v2  AC_alt_v1  AC_alt_v2   AN_v1   AN_v2 is_indel  MAF_ratio
        51573230:T:C    19 51573230         T   C v2_only          N          Y      NaN 0.000011          NaN          1.0        NaN        1.0     NaN 87434.0    False        NaN
       51573236:TG:T    19 51573236        TG   T v2_only          N          Y      NaN 0.000000          NaN          0.0        NaN        0.0     NaN 87454.0     True        NaN
        51573255:C:G    19 51573255         C   G v2_only          N          Y      NaN 0.000011          NaN          1.0        NaN        1.0     NaN 87462.0    False        NaN
51573255:CCTATCCAG:C    19 51573255 CCTATCCAG   C  shared          Y          Y 0.000044 0.000011          1.0          1.0        1.0        1.0 22898.0 87462.0     True   0

## 2. Variantes "v1-only" — sumiram no v2 (o grupo mais informativo)

Estas são as candidatas a **artefato técnico**: presentes nos 11K, ausentes nos 44K.
Se forem **indels**, parte pode ser apenas **representação diferente** (sem left-align) — sinalizamos para checar.

In [5]:
# --- v1-only detail ---
cols = ["key","POS_v1","REF_v1","ALT_v1","MAF_v1","carriers_v1","AC_alt_v1","AN_v1","is_indel_v1"]
v1o = v1_only[cols].sort_values("carriers_v1", ascending=False)
print(f"v1-only variants: {len(v1o)}")
print(v1o.head(20).to_string(index=False))
v1o.to_csv(RESULTS / "cmp_v1_only.csv", index=False)

n_indel = int(v1_only["is_indel_v1"].fillna(False).sum())
if n_indel > 0:
    print(f"\nNOTE: {n_indel} of the v1-only are indels -> if many, re-run extraction with "
          "`bcftools norm -f <GRCh38.fa>` (left-align) before trusting 'disappeared'.")

v1-only variants: 6
         key     POS_v1 REF_v1 ALT_v1   MAF_v1  carriers_v1  AC_alt_v1   AN_v1 is_indel_v1
51587731:G:* 51587731.0      G      * 0.000131          3.0        3.0 22902.0       False
51587677:G:A 51587677.0      G      A 0.000087          2.0        2.0 22900.0       False
51573274:C:G 51573274.0      C      G 0.000044          1.0        1.0 22898.0       False
51573263:G:* 51573263.0      G      * 0.000044          1.0        1.0 22898.0       False
51581354:G:C 51581354.0      G      C 0.000044          1.0        1.0 22900.0       False
51588382:C:G 51588382.0      C      G 0.000044          1.0        1.0 22902.0       False


## 3. Variantes compartilhadas — o MAF é estável?

Para as presentes nas duas, comparamos MAF_v1 × MAF_v2 (escala log).
Estabilidade (pontos na diagonal) → substrato estável. Deslocamentos grandes → investigar.

In [6]:
# --- Shared MAF comparison ---
s = shared[["key","POS_v1","REF_v1","ALT_v1","is_indel_v1",
            "MAF_v1","MAF_v2","carriers_v1","carriers_v2","AN_v1","AN_v2"]].copy()
s["MAF_ratio"] = s["MAF_v2"] / s["MAF_v1"].replace(0, np.nan)
s["abs_log2_ratio"] = np.abs(np.log2(s["MAF_ratio"]))
corr = s[["MAF_v1","MAF_v2"]].corr().iloc[0,1]
print(f"shared variants: {len(s)}")
print(f"Pearson corr(MAF_v1, MAF_v2): {corr:.4f}")

print("\nLargest MAF shifts (shared):")
print(s.sort_values("abs_log2_ratio", ascending=False)
       .head(12)[["key","MAF_v1","MAF_v2","MAF_ratio","carriers_v1","carriers_v2"]]
       .to_string(index=False))
s.sort_values("POS_v1").to_csv(RESULTS / "cmp_shared_maf.csv", index=False)

# scatter plot
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(5.5,5.5))
    ax.scatter(s["MAF_v1"], s["MAF_v2"], s=18, alpha=0.6)
    lims = [max(1e-5, min(s["MAF_v1"].min(), s["MAF_v2"].min())*0.5), 1]
    ax.plot(lims, lims, "r--", lw=1, label="y=x")
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel("MAF v1 (Park / Freeze One, 11.451)")
    ax.set_ylabel("MAF v2 (Hui / Release 2020 v2.0, 43.731)")
    ax.set_title("ZNF175 shared-variant MAF: v1 vs v2")
    ax.legend()
    fig.tight_layout()
    fig.savefig(RESULTS / "cmp_shared_maf_scatter.png", dpi=120)
    print("\nsaved -> results/cmp_shared_maf_scatter.png")
except Exception as e:
    print("plot skipped:", e)

shared variants: 155
Pearson corr(MAF_v1, MAF_v2): 0.9449

Largest MAF shifts (shared):
                 key   MAF_v1   MAF_v2  MAF_ratio  carriers_v1  carriers_v2
        51581871:A:G 0.000044 0.000000   0.000000          1.0          0.0
        51588539:G:T 0.000109 0.009296  85.094469          2.0        800.0
        51588529:A:C 0.010020 0.128170  12.790779        149.0      10366.0
        51573263:G:A 0.000087 0.000011   0.130902          2.0          1.0
        51581982:G:A 0.000131 0.000023   0.174502          3.0          2.0
        51588493:A:T 0.000044 0.000011   0.257711          1.0          1.0
        51581458:C:T 0.000044 0.000011   0.261805          1.0          1.0
        51573414:A:G 0.000044 0.000011   0.261805          1.0          1.0
        51581490:G:T 0.000044 0.000011   0.261805          1.0          1.0
51573255:CCTATCCAG:C 0.000044 0.000011   0.261805          1.0          1.0
        51573313:A:G 0.000044 0.000011   0.261805          1.0          1.0


/project/hall/analysis/hearing-loss-genomics/venv/lib/python3.12/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log2
  result = getattr(ufunc, method)(*inputs, **kwargs)



saved -> results/cmp_shared_maf_scatter.png


## 4. Cross-check: v1 (pipeline bcftools) × v1 (NB 01, plink/PLINK)

Sanidade: o v1 extraído por dois caminhos diferentes deve concordar.

In [7]:
# --- Cross-check v1 strict vs NB01 plink-PLINK ---
try:
    v1_plink = load_freq(RESULTS01 / "v1_znf175_freeze_one")
    x = v1.merge(v1_plink, on="key", how="outer", suffixes=("_strict","_plink"), indicator=True)
    print("v1 strict (bcftools):", len(v1), "| v1 plink-PLINK (NB01):", len(v1_plink))
    print(x["_merge"].value_counts().rename({"both":"shared","left_only":"strict-only","right_only":"plink-only"}).to_string())
    sc = x[x["_merge"]=="both"]
    if len(sc):
        print("max |MAF_strict - MAF_plink| on shared:",
              float((sc["MAF_strict"]-sc["MAF_plink"]).abs().max()))
except Exception as e:
    print("cross-check skipped:", e)

v1 strict (bcftools): 161 | v1 plink-PLINK (NB01): 151
_merge
shared         151
strict-only     10
plink-only       0
max |MAF_strict - MAF_plink| on shared: 0.0


In [8]:
# --- Summary ---
n_star    = int((v1_only["ALT_v1"] == "*").sum())
v1_rare   = int((v1["MAF"] < 0.001).sum());  v1_common = int((v1["MAF"] >= 0.01).sum())
v2_rare   = int((v2["MAF"] < 0.001).sum())
summary = f'''# NB 02 — ZNF175 MAF comparison: v1 (Park) vs v2 (Hui)

Window: chr19:51,571,283-51,592,510 (NCBI NC_000019.10, GRCh38, + strand).
Pipeline (symmetric, scientist-independent): bcftools view -r (.tbi random access)
-> norm -f <GRCh38> -a -m-any (left-align + atomize + split) -> plink2 --freq --geno-counts.

## Counts (whole-cohort)
- v1 (Freeze One, 11,451):       {len(v1)} variants ({v1_rare} rare <0.1%, {v1_common} common >=1%)
- v2 (Release 2020 v2.0, 43,731): {len(v2)} variants ({v2_rare} rare <0.1%)
- UNION: {len(u)}  ->  shared {len(shared)} | v1-only {len(v1_only)} | v2-only {len(v2_only)}
- Pearson corr(MAF_v1, MAF_v2) on shared: {corr:.4f}

## Scientific reading
The ZNF175 variant SUBSTRATE is stable between v1 and v2. The variants underpinning the original
signal are still present in v2 with correlated MAF (r={corr:.2f}). The apparent differences are explained:
- v2-only ({len(v2_only)}): EXPECTED -- 4x more samples reveal more rare variants.
- v1-only ({len(v1_only)}): NOT real disappearances -- {n_star} are '*' spanning-deletion markers
  (representation artifacts) + {len(v1_only)-n_star} ultra-rare private singletons (1-2 carriers).
- the {v1_common} common (>=1%) variants are identical across freezes (stable backbone).

=> The 11K->45K signal loss is NOT explained by variant-substrate instability. It points instead to
   (a) winner's curse (a small-case signal regressing as N grows), or
   (b) association-pipeline differences flagged in the 2026-06-03 meeting (replications shifting to
       missense vs the pLOF discovery; possibly different phenotype-release versions).
CAVEAT: this step compares MAF; it does NOT re-test the association. It rules out ONE technical
        explanation (substrate). A definitive verdict needs the association re-run with matched pipeline.

## Outputs
- cmp_union_all_variants.csv  (UNION -- all {len(u)} variants, blanks where absent in a cohort)
- cmp_shared_maf.csv, cmp_v1_only.csv, cmp_shared_maf_scatter.png, per_cohort_overview.csv

## Next steps
1. Annotate the union table with BF4 -> pLOF status + AlphaMissense scores, to focus on the variants
   that actually enter the burden (rare pLOF / deleterious missense).
2. Stratify MAF by ancestry (removes the remaining whole-cohort confounder: cohort size & ancestry mix).
3. (definitive) re-run the ZNF175-tinnitus association on v1 and v2 with a matched pipeline.
'''
(RESULTS / "nb02_comparison_summary.md").write_text(summary)
print(summary)

# NB 02 — ZNF175 MAF comparison: v1 (Park) vs v2 (Hui)

Window: chr19:51,571,283-51,592,510 (NCBI NC_000019.10, GRCh38, + strand).
Pipeline (symmetric, scientist-independent): bcftools view -r (.tbi random access)
-> norm -f <GRCh38> -a -m-any (left-align + atomize + split) -> plink2 --freq --geno-counts.

## Counts (whole-cohort)
- v1 (Freeze One, 11,451):       161 variants (142 rare <0.1%, 8 common >=1%)
- v2 (Release 2020 v2.0, 43,731): 380 variants (356 rare <0.1%)
- UNION: 386  ->  shared 155 | v1-only 6 | v2-only 225
- Pearson corr(MAF_v1, MAF_v2) on shared: 0.9449

## Scientific reading
The ZNF175 variant SUBSTRATE is stable between v1 and v2. The variants underpinning the original
signal are still present in v2 with correlated MAF (r=0.94). The apparent differences are explained:
- v2-only (225): EXPECTED -- 4x more samples reveal more rare variants.
- v1-only (6): NOT real disappearances -- 2 are '*' spanning-deletion markers
  (representation artifacts) + 4 ultra-rare privat